# RFD Urban Digital Twin Analysis

This notebook builds compact exam pipeline for RFD-based consistency profiling on urban air-quality data.

Scope:
- simplified conceptual Urban Digital Twin;
- Beijing air-quality subset with two stations;
- interpretable approximate rules, not causal claims;
- lightweight discovery over reduced deterministic sample.


## Dataset Configuration

Selected subset:
- stations: `Aotizhongxin`, `Changping`
- period: `2013-09-01` to `2013-12-31`
- cleaned dataset target: around 5k-6k rows
- RFD experiment sample: 1500 rows balanced across stations

Why December included: Sep-Nov cleaned subset produced only 3996 rows after dropping missing values. Adding December yields 5426 rows.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [current, *current.parents] if (path / 'AGENTS.md').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

for path in [DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [2]:
from src.experiments import (
    prepare_rfd_sample,
    run_candidate_validation,
    run_lightweight_discovery,
    run_station_comparison,
    run_threshold_comparison,
    select_top_rule_violations,
    export_violation_examples,
    run_candidate_metrics,
    run_bootstrap_validation,
    run_train_test_validation,
    run_window_evolution,
    run_violation_analysis,
)
from src.preprocessing import prepare_udt_dataset
from src.profiling import (
    correlation_matrix,
    data_types_summary,
    dataset_shape_summary,
    hour_distribution,
    missing_values_summary,
    numeric_summary,
    station_distribution,
    time_slot_distribution,
    unique_values_summary,
)
from src.visualization import (
    plot_correlation_matrix,
    plot_missing_values,
    plot_pm25_timeseries,
    plot_pollutant_distribution,
)


## Step 1-2: Preprocessing

In [3]:
cleaned_df, pre_drop_missing = prepare_udt_dataset(
    raw_dir=DATA_RAW,
    processed_path=DATA_PROCESSED / 'udt_rfd_dataset.csv',
)
sample_df = prepare_rfd_sample(cleaned_df)
sample_df.to_csv(DATA_PROCESSED / 'udt_rfd_sample.csv', index=False)

print('Cleaned rows:', len(cleaned_df))
print('Cleaned columns:', cleaned_df.shape[1])
print('RFD sample rows:', len(sample_df))
display(pre_drop_missing)


Cleaned rows: 5426
Cleaned columns: 11
RFD sample rows: 1500


,column,missing_count,missing_rate
0,O3,390,0.066598
1,NO2,32,0.005464
2,PM2.5,7,0.001195
3,PM10,3,0.000512
4,DEWP,0,0.000000
5,TEMP,0,0.000000
6,WSPM,0,0.000000
7,datetime,0,0.000000
8,hour,0,0.000000
9,station,0,0.000000


## Step 3: Profiling

In [4]:
shape_df = dataset_shape_summary(cleaned_df)
dtypes_df = data_types_summary(cleaned_df)
missing_df = missing_values_summary(cleaned_df)
unique_df = unique_values_summary(cleaned_df)
numeric_df = numeric_summary(cleaned_df)
station_df = station_distribution(cleaned_df)
hour_df = hour_distribution(cleaned_df)
time_slot_df = time_slot_distribution(cleaned_df)
corr_df = correlation_matrix(
    cleaned_df,
    numeric_columns=['hour', 'PM2.5', 'PM10', 'NO2', 'O3', 'TEMP', 'DEWP', 'WSPM'],
)

shape_df.to_csv(RESULTS_DIR / 'profile_shape_summary.csv', index=False)
dtypes_df.to_csv(RESULTS_DIR / 'profile_dtypes_summary.csv', index=False)
missing_df.to_csv(RESULTS_DIR / 'profile_missing_summary.csv', index=False)
unique_df.to_csv(RESULTS_DIR / 'profile_unique_values_summary.csv', index=False)
numeric_df.to_csv(RESULTS_DIR / 'profile_numeric_summary.csv', index=False)
station_df.to_csv(RESULTS_DIR / 'profile_station_distribution.csv', index=False)
hour_df.to_csv(RESULTS_DIR / 'profile_hour_distribution.csv', index=False)
time_slot_df.to_csv(RESULTS_DIR / 'profile_time_slot_distribution.csv', index=False)
corr_df.to_csv(RESULTS_DIR / 'profile_correlation_matrix.csv')

plot_missing_values(missing_df, FIGURES_DIR / 'missing_values.png')
plot_pollutant_distribution(cleaned_df, 'PM2.5', FIGURES_DIR / 'pm25_distribution.png')
plot_pollutant_distribution(cleaned_df, 'NO2', FIGURES_DIR / 'no2_distribution.png')
plot_pm25_timeseries(cleaned_df, FIGURES_DIR / 'pm25_timeseries.png')
plot_correlation_matrix(corr_df, FIGURES_DIR / 'correlation_matrix.png')

print('Dataset shape')
display(shape_df)
print('Data types')
display(dtypes_df)
print('Missing values after cleaning')
display(missing_df)
print('Unique values')
display(unique_df)
print('Numeric summary')
display(numeric_df)
print('Station distribution')
display(station_df)
print('Hour distribution')
display(hour_df)
print('Time-slot distribution')
display(time_slot_df)


Dataset shape


,metric,value
0,rows,5426
1,columns,11


Data types


,column,dtype
0,DEWP,float64
1,NO2,float64
2,O3,float64
3,PM10,float64
4,PM2.5,float64
5,TEMP,float64
6,WSPM,float64
7,datetime,datetime64[us]
8,hour,int64
9,station,str


Missing values after cleaning


...

,station,rows
0,Aotizhongxin,2548
1,Changping,2878


Hour distribution


,hour,rows
0,0,220
1,1,224
2,2,217
3,3,212
4,4,205
5,5,222
6,6,224
7,7,221
8,8,217
9,9,228


Time-slot distribution


,time_slot,rows
0,night,1300
1,morning,1357
2,afternoon,1422
3,evening,1347


## Step 4-6: Candidate RFD Validation

Threshold sets are defined in `src/rfd.py` as `strict`, `medium`, and `relaxed`.
Candidate rules are evaluated on balanced 1500-row sample to keep pairwise validation tractable.


In [5]:
candidate_df, candidate_violations = run_candidate_validation(
    sample_df,
    RESULTS_DIR / 'rfd_candidate_results.csv',
)

candidate_table = candidate_df.loc[:, ['rule_label', 'support', 'confidence', 'violations', 'interpretation']].copy()
for column in ['support', 'confidence']:
    candidate_table[column] = candidate_table[column].round(4)

display(candidate_table)


,rule_label,support,confidence,violations,interpretation
2,"station, PM2.5, NO2 -> PM10",0.0292,0.5204,15729,"Within one station, similar PM2.5 and NO2 shou..."
1,"station, time_slot, PM2.5 -> PM10",0.0191,0.4770,11208,"Within one station and time slot, similar PM2...."
5,"station, time_slot, NO2 -> O3",0.0268,0.4728,15892,"Within one station and time slot, similar NO2 ..."
0,"station, PM2.5 -> PM10",0.0734,0.4663,44020,"Within one station, particulate measures shoul..."
3,"station, hour, TEMP, WSPM -> O3",0.0053,0.4419,3343,"Within one station and similar hour, temperatu..."
4,"station, time_slot, TEMP, WSPM -> O3",0.0103,0.4252,6627,"Within one station and time slot, similar temp..."
6,"station, TEMP, DEWP, WSPM -> PM2.5",0.0098,0.1783,9048,"Within one station, similar meteorological con..."


In [6]:
candidate_metrics_df = run_candidate_metrics(
    sample_df,
    RESULTS_DIR / 'rfd_candidate_metrics.csv',
    FIGURES_DIR / 'rfd_lift_vs_baseline.png',
)

metrics_table = candidate_metrics_df.loc[:, [
    'representation', 'rule_label', 'support', 'confidence',
    'baseline_confidence', 'lift', 'violation_rate'
]].copy()
for column in ['support', 'confidence', 'baseline_confidence', 'lift', 'violation_rate']:
    metrics_table[column] = metrics_table[column].round(4)

display(metrics_table)


,representation,rule_label,support,confidence,baseline_confidence,lift,violation_rate
0,raw,"station, PM2.5, NO2 -> PM10",0.0292,0.5204,0.1470,3.5415,0.4796
1,raw,"station, time_slot, PM2.5 -> PM10",0.0191,0.4770,0.1482,3.2189,0.5230
2,raw,"station, PM2.5 -> PM10",0.0734,0.4663,0.1478,3.1549,0.5337
3,raw,"station, time_slot, NO2 -> O3",0.0268,0.4728,0.2719,1.7389,0.5272
4,raw,"station, hour, TEMP, WSPM -> O3",0.0053,0.4419,0.2701,1.6363,0.5581
5,raw,"station, time_slot, TEMP, WSPM -> O3",0.0103,0.4252,0.2712,1.5675,0.5748
6,raw,"station, TEMP, DEWP, WSPM -> PM2.5",0.0098,0.1783,0.1465,1.2168,0.8217
7,binned,"station, PM2.5_bin, NO2_bin -> PM10_bin",0.0930,0.7336,0.3325,2.2066,0.2664
8,binned,"station, time_slot, PM2.5_bin -> PM10_bin",0.0425,0.6565,0.3329,1.9721,0.3435
9,binned,"station, PM2.5_bin -> PM10_bin",0.1667,0.6440,0.3327,1.9354,0.3560


## Step 7: Lightweight Discovery

In [7]:
discovery_df = run_lightweight_discovery(
    sample_df,
    RESULTS_DIR / 'rfd_discovered_top10.csv',
)

if discovery_df.empty:
    print('No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.')
else:
    display(discovery_df.round(4))


No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.


## Step 8-9: Threshold and Station Comparisons

In [8]:
threshold_df = run_threshold_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_threshold_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_threshold.png',
)
station_df_rfd = run_station_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_station_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_station.png',
)

display(threshold_df[['rule_label', 'threshold_set', 'support', 'confidence', 'violations']].round(4))
display(station_df_rfd[['rule_label', 'station_scope', 'support', 'confidence', 'violations']].round(4))


,rule_label,threshold_set,support,confidence,violations
0,"station, PM2.5 -> PM10",medium,0.0734,0.4663,44020
1,"station, PM2.5 -> PM10",relaxed,0.1004,0.6104,43983
2,"station, PM2.5 -> PM10",strict,0.0417,0.3741,29367
3,"station, PM2.5, NO2 -> PM10",medium,0.0292,0.5204,15729
4,"station, PM2.5, NO2 -> PM10",relaxed,0.0522,0.6652,19648
5,"station, PM2.5, NO2 -> PM10",strict,0.0099,0.4217,6461
6,"station, TEMP, DEWP, WSPM -> PM2.5",medium,0.0098,0.1783,9048
7,"station, TEMP, DEWP, WSPM -> PM2.5",relaxed,0.0244,0.2429,20743
8,"station, TEMP, DEWP, WSPM -> PM2.5",strict,0.0019,0.1267,1895
9,"station, hour, TEMP, WSPM -> O3",medium,0.0053,0.4419,3343


,rule_label,station_scope,support,confidence,violations
0,PM2.5 -> PM10,Aotizhongxin,0.1430,0.4366,22633
1,PM2.5 -> PM10,Changping,0.1507,0.4946,21387
2,"PM2.5, NO2 -> PM10",Aotizhongxin,0.0552,0.5064,7648
3,"PM2.5, NO2 -> PM10",Changping,0.0616,0.5330,8081
4,"TEMP, DEWP, WSPM -> PM2.5",Aotizhongxin,0.0215,0.1769,4979
5,"TEMP, DEWP, WSPM -> PM2.5",Changping,0.0177,0.1800,4069
6,"hour, TEMP, WSPM -> O3",Aotizhongxin,0.0111,0.4813,1622
7,"hour, TEMP, WSPM -> O3",Changping,0.0102,0.3989,1721
8,"time_slot, NO2 -> O3",Aotizhongxin,0.0487,0.5256,6487
9,"time_slot, NO2 -> O3",Changping,0.0587,0.4291,9405


In [9]:
bootstrap_df = run_bootstrap_validation(
    cleaned_df,
    RESULTS_DIR / 'rfd_bootstrap_summary.csv',
    RESULTS_DIR / 'rfd_bootstrap_iterations.csv',
)
train_test_df = run_train_test_validation(
    cleaned_df,
    RESULTS_DIR / 'rfd_train_test_validation.csv',
)
window_df = run_window_evolution(
    cleaned_df,
    candidate_metrics_df,
    RESULTS_DIR / 'rfd_window_evolution.csv',
    FIGURES_DIR / 'rfd_confidence_over_time.png',
)
violations_summary_df = run_violation_analysis(
    sample_df,
    candidate_metrics_df,
    RESULTS_DIR / 'rfd_violations_summary.csv',
    RESULTS_DIR / 'rfd_top_violating_pairs.csv',
    FIGURES_DIR / 'rfd_violations_by_month_station.png',
)

print('Bootstrap summary')
display(bootstrap_df.round(4))
print('Temporal train-test validation')
display(train_test_df.round(4))
print('Window evolution')
display(window_df.loc[:, [
    'rule_label', 'window_start', 'support', 'confidence', 'violation_rate',
    'baseline_confidence', 'lift', 'abrupt_change'
]].round(4))
print('Violation concentration')
display(violations_summary_df.head(20).round(4))


Bootstrap summary


,rule_label,lhs,rhs,support_mean,support_std,support_ci95_low,support_ci95_high,confidence_mean,confidence_std,confidence_ci95_low,confidence_ci95_high,lift_mean,lift_std,lift_ci95_low,lift_ci95_high,iterations
0,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,0.0304,0.0021,0.0270,0.0343,0.5199,0.0163,0.4936,0.5498,3.4081,0.1263,3.2128,3.6280,30
1,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,0.0196,0.0009,0.0183,0.0213,0.4847,0.0144,0.4644,0.5138,3.1881,0.1164,3.0021,3.4680,30
2,"station, PM2.5 -> PM10","station, PM2.5",PM10,0.0756,0.0035,0.0703,0.0824,0.4716,0.0139,0.4517,0.4955,3.0919,0.1101,2.9305,3.2951,30
3,"station, time_slot, NO2 -> O3","station, time_slot, NO2",O3,0.0269,0.0006,0.0259,0.0279,0.4955,0.0155,0.4666,0.5180,1.7745,0.0421,1.7101,1.8539,30
4,"station, hour, TEMP, WSPM -> O3","station, hour, TEMP, WSPM",O3,0.0054,0.0002,0.0051,0.0058,0.4712,0.0180,0.4388,0.5055,1.6929,0.0542,1.5998,1.7949,30
5,"station, time_slot, TEMP, WSPM -> O3","station, time_slot, TEMP, WSPM",O3,0.0102,0.0004,0.0094,0.0108,0.4506,0.0165,0.4228,0.4793,1.6125,0.0422,1.5435,1.6921,30
6,"station, TEMP, DEWP, WSPM -> PM2.5","station, TEMP, DEWP, WSPM",PM2.5,0.0102,0.0006,0.0090,0.0112,0.2003,0.0128,0.1790,0.2219,1.3314,0.0640,1.2371,1.4303,30


Temporal train-test validation


,rule_label,lhs,rhs,support_train,confidence_train,violation_rate_train,baseline_confidence_train,lift_train,support_test,confidence_test,violation_rate_test,baseline_confidence_test,lift_test,confidence_delta_test_minus_train,lift_delta_test_minus_train
0,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,0.0273,0.4903,0.5097,0.1450,3.3825,0.0434,0.5776,0.4224,0.1654,3.4914,0.0873,0.1089
1,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,0.0183,0.4617,0.5383,0.1444,3.1966,0.0228,0.5246,0.4754,0.1654,3.1718,0.0629,-0.0248
2,"station, PM2.5 -> PM10","station, PM2.5",PM10,0.0711,0.4419,0.5581,0.1452,3.0445,0.0873,0.5097,0.4903,0.1642,3.1032,0.0678,0.0588
3,"station, time_slot, NO2 -> O3","station, time_slot, NO2",O3,0.0266,0.4472,0.5528,0.2593,1.7250,0.0265,0.7323,0.2677,0.3554,2.0606,0.2851,0.3356
4,"station, hour, TEMP, WSPM -> O3","station, hour, TEMP, WSPM",O3,0.0074,0.4199,0.5801,0.2580,1.6280,0.0102,0.5731,0.4269,0.3531,1.6227,0.1531,-0.0053
5,"station, time_slot, TEMP, WSPM -> O3","station, time_slot, TEMP, WSPM",O3,0.0142,0.4070,0.5930,0.2587,1.5733,0.0186,0.5599,0.4401,0.3545,1.5793,0.1529,0.0060
6,"station, TEMP, DEWP, WSPM -> PM2.5","station, TEMP, DEWP, WSPM",PM2.5,0.0146,0.1674,0.8326,0.1422,1.1765,0.0191,0.2588,0.7412,0.1737,1.4903,0.0914,0.3138


Window evolution


/var/folders/hn/yqq1wz1x1qg3gbdk_f7rxb5c0000gn/T/ipykernel_79624/2356862301.py:32: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(4))


,rule_label,window_start,support,confidence,violation_rate,baseline_confidence,lift,abrupt_change
0,"station, PM2.5 -> PM10",2013-09-01,0.0641,0.4191,0.5809,0.1512,2.7727,False
1,"station, PM2.5 -> PM10",2013-10-01,0.0592,0.4750,0.5250,0.1282,3.7062,True
2,"station, PM2.5 -> PM10",2013-11-01,0.1214,0.5120,0.4880,0.1960,2.6118,True
3,"station, PM2.5 -> PM10",2013-12-01,0.0873,0.5097,0.4903,0.1649,3.0906,False
4,"station, PM2.5, NO2 -> PM10",2013-09-01,0.0222,0.4977,0.5023,0.1511,3.2932,False
5,"station, PM2.5, NO2 -> PM10",2013-10-01,0.0217,0.5104,0.4896,0.1269,4.0220,True
6,"station, PM2.5, NO2 -> PM10",2013-11-01,0.0662,0.5218,0.4782,0.1908,2.7346,True
7,"station, PM2.5, NO2 -> PM10",2013-12-01,0.0434,0.5776,0.4224,0.1646,3.5083,True
8,"station, time_slot, NO2 -> O3",2013-09-01,0.0336,0.3187,0.6813,0.1932,1.6490,False
9,"station, time_slot, NO2 -> O3",2013-10-01,0.0268,0.5368,0.4632,0.3031,1.7708,True


Violation concentration


,rule_label,station,month,time_slot,violation_pairs,mean_rhs_abs_diff,max_rhs_abs_diff,share_within_rule
0,"station, PM2.5, NO2 -> PM10",Changping,2013-09,morning,29,214.1034,253.0,0.145
1,"station, PM2.5, NO2 -> PM10",Changping,2013-09,night,20,215.1000,279.0,0.100
2,"station, PM2.5, NO2 -> PM10",Changping,2013-12,afternoon,16,220.5000,268.0,0.080
3,"station, PM2.5, NO2 -> PM10",Changping,2013-09,afternoon,15,199.8000,247.0,0.075
4,"station, PM2.5, NO2 -> PM10",Changping,2013-12,evening,12,202.8333,222.0,0.060
5,"station, PM2.5, NO2 -> PM10",Changping,2013-10,night,11,205.6364,223.0,0.055
6,"station, PM2.5, NO2 -> PM10",Changping,2013-10,afternoon,9,194.6667,216.0,0.045
7,"station, PM2.5, NO2 -> PM10",Changping,2013-12,night,8,211.8750,243.0,0.040
8,"station, PM2.5, NO2 -> PM10",Changping,2013-09,evening,8,202.1250,248.0,0.040
9,"station, PM2.5, NO2 -> PM10",Changping,2013-11,afternoon,7,199.1429,216.0,0.035


## Step 10: Violation Analysis

In [10]:
selected_violations = select_top_rule_violations(candidate_df, candidate_violations, top_rules=2, per_rule=5)
violations_df = export_violation_examples(
    selected_violations,
    RESULTS_DIR / 'violations_examples.csv',
    limit=10,
)

display(violations_df)


,rule_label,lhs,rhs,row_index_1,row_index_2,datetime_1,datetime_2,station_1,station_2,rhs_value_1,rhs_value_2,rhs_abs_diff,PM2.5_1,PM2.5_2,NO2_1,NO2_2,time_slot_1,time_slot_2
0,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,36,167,2013-09-01 18:00:00,2013-09-04 12:00:00,Aotizhongxin,Aotizhongxin,103.0,44.0,59.0,43.0,34.0,53.0,49.0,NaN,NaN
1,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,36,236,2013-09-01 18:00:00,2013-09-05 23:00:00,Aotizhongxin,Aotizhongxin,103.0,44.0,59.0,43.0,39.0,53.0,43.0,NaN,NaN
2,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,36,245,2013-09-01 18:00:00,2013-09-06 04:00:00,Aotizhongxin,Aotizhongxin,103.0,53.0,50.0,43.0,43.0,53.0,52.0,NaN,NaN
3,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,36,247,2013-09-01 18:00:00,2013-09-06 05:00:00,Aotizhongxin,Aotizhongxin,103.0,46.0,57.0,43.0,45.0,53.0,48.0,NaN,NaN
4,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,36,255,2013-09-01 18:00:00,2013-09-06 09:00:00,Aotizhongxin,Aotizhongxin,103.0,50.0,53.0,43.0,45.0,53.0,57.0,NaN,NaN
5,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,36,236,2013-09-01 18:00:00,2013-09-05 23:00:00,Aotizhongxin,Aotizhongxin,103.0,44.0,59.0,43.0,39.0,NaN,NaN,evening,evening
6,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,36,1442,2013-09-01 18:00:00,2013-10-01 22:00:00,Aotizhongxin,Aotizhongxin,103.0,57.0,46.0,43.0,41.0,NaN,NaN,evening,evening
7,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,36,2509,2013-09-01 18:00:00,2013-10-24 21:00:00,Aotizhongxin,Aotizhongxin,103.0,50.0,53.0,43.0,38.0,NaN,NaN,evening,evening
8,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,36,2741,2013-09-01 18:00:00,2013-10-29 21:00:00,Aotizhongxin,Aotizhongxin,103.0,123.0,20.0,43.0,50.0,NaN,NaN,evening,evening
9,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,36,3988,2013-09-01 18:00:00,2013-11-30 20:00:00,Aotizhongxin,Aotizhongxin,103.0,76.0,27.0,43.0,39.0,NaN,NaN,evening,evening


## Takeaways

- Candidate RFDs are interpretable but moderately weak on this subset; none behaves like near-deterministic rule.
- Relaxed thresholds increase confidence and support, but still do not yield discovery results above project filter (`0.01`, `0.85`).
- Violating pairs remain useful for UDT framing: they can indicate local inconsistency, unusual pollution dynamics, or missing contextual variables.
- This is approximate consistency analysis, not prediction and not causality.
